# Travel Reimbursement Approval Agent
### HCLTech — GenAI Developer / Architect Technical Assignment

A lightweight, policy-grounded Agentic AI prototype for evaluating travel reimbursement claims. It uses tool orchestration, deterministic business guardrails, structured output validation, audit evidence, and a results dashboard.

## Architecture

**Claim → Agent Planner → Policy Retrieval / Receipt / Limit / Approval Tools → Decision Guardrails → Pydantic Validation → JSON → Dashboard**

The implementation is intentionally small and reviewable. The optional LLM planner can use an OpenAI-compatible API; without a key, the notebook uses a deterministic fallback so it still runs end-to-end.

In [ ]:
# Optional installation cell for a fresh Jupyter/Colab environment
# %pip install -q pandas pydantic matplotlib openai

import os, json, re
from datetime import datetime
from typing import Any, Dict, List, Literal
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pydantic import BaseModel, Field

## 1. Policy Knowledge Base

The policy below mirrors the assignment's supplied rules. A lightweight retrieval function provides relevant policy context before decisions are made. For this small policy set, keyword retrieval is preferable to unnecessary vector-database infrastructure.

In [ ]:
POLICY_DOCS = [
 {"id":"POL-CAT-01","text":"Eligible: economy airfare, lodging, meals subject to per-diem, ground transport, conference/registration fees for documented business purposes."},
 {"id":"POL-CAT-02","text":"Ineligible: alcohol, minibar, spa, gym, personal entertainment, in-room movies, personal shopping, gifts, traffic fines, penalties, late fees and personal expenses."},
 {"id":"POL-PD-01","text":"Meals maximum 75 USD per day; excess is deducted."},
 {"id":"POL-PD-02","text":"Lodging maximum 200 USD per night; excess is deducted."},
 {"id":"POL-PD-03","text":"Ground transport maximum 50 USD per day; excess is deducted."},
 {"id":"POL-AIR-01","text":"Only economy airfare is reimbursable. Business/first class is an exception and must go to Manual Review."},
 {"id":"POL-RCT-01","text":"Any single item above 25 USD requires an attached itemized receipt. Airfare and lodging always require a receipt."},
 {"id":"POL-RCT-02","text":"Missing required receipt routes the claim to Manual Review."},
 {"id":"POL-APR-01","text":"Total reimbursable amount <= 500 USD may be auto-approved when fully compliant."},
 {"id":"POL-APR-02","text":"Total reimbursable amount > 500 and <= 2000 USD is approvable when fully compliant."},
 {"id":"POL-APR-03","text":"Total reimbursable amount > 2000 USD must go to Manual Review."},
 {"id":"POL-TIME-01","text":"Claims must be submitted within 30 days of the expense date; late claims go to Manual Review."}
]

def retrieve_policy(query: str, top_k: int = 4):
    q=set(re.findall(r"[a-z0-9]+", query.lower()))
    scored=[]
    for d in POLICY_DOCS:
        t=set(re.findall(r"[a-z0-9]+", d["text"].lower()))
        score=len(q&t)
        if score: scored.append((score,d))
    scored.sort(key=lambda x:x[0], reverse=True)
    return [d for _,d in scored[:top_k]]

## 2. Assignment Claims

Only the five claims supplied in the assignment are evaluated.

In [ ]:
CLAIMS=[
 {"claim_id":"CLM-001","employee":"A. Rivera","trip_end":"2026-06-12","submitted":"2026-06-20","items":[
  {"category":"airfare","description":"Round-trip economy airfare","amount":420,"receipt_attached":True},
  {"category":"lodging","description":"Hotel, 2 nights @ $180","amount":360,"receipt_attached":True,"nights":2},
  {"category":"meals","description":"Meals, 3 days @ ~$60/day","amount":180,"receipt_attached":True,"days":3},
  {"category":"conference_fees","description":"Conference registration","amount":150,"receipt_attached":True}]},
 {"claim_id":"CLM-002","employee":"B. Osei","trip_end":"2026-06-15","submitted":"2026-06-25","items":[
  {"category":"spa","description":"Hotel spa package","amount":300,"receipt_attached":True},
  {"category":"minibar","description":"In-room minibar","amount":80,"receipt_attached":True}]},
 {"claim_id":"CLM-003","employee":"C. Nakamura","trip_end":"2026-06-10","submitted":"2026-06-22","items":[
  {"category":"airfare","description":"Round-trip economy airfare","amount":300,"receipt_attached":True},
  {"category":"lodging","description":"Hotel, 2 nights @ $250","amount":500,"receipt_attached":True,"nights":2},
  {"category":"meals","description":"Meals, 2 days @ $70/day","amount":140,"receipt_attached":True,"days":2}]},
 {"claim_id":"CLM-004","employee":"D. Fischer","trip_end":"2026-06-18","submitted":"2026-06-28","items":[
  {"category":"airfare","description":"Business-class international airfare","amount":2400,"receipt_attached":True,"class":"business"},
  {"category":"lodging","description":"Hotel, 3 nights","amount":600,"receipt_attached":False,"nights":3}]},
 {"claim_id":"CLM-005","employee":"E. Haddad","trip_end":"2026-06-11","submitted":"2026-06-24","items":[
  {"category":"meals","description":"Client dinner for 4 (business development)","amount":220,"receipt_attached":False,"days":1}]}
]

## 3. Agent Tools

Four meaningful tools are exposed to the agent: policy retrieval, receipt validation, category-limit checking, and approval-threshold checking.

In [ ]:
def policy_lookup(item):
    return {"tool":"policy_lookup","policies":[d for d in retrieve_policy(item["category"]+" "+item["description"])]}

def receipt_checker(item):
    required=item["category"] in {"airfare","lodging"} or float(item["amount"])>25
    missing=required and not item.get("receipt_attached",False)
    return {"tool":"receipt_checker","required":required,"attached":item.get("receipt_attached",False),"missing":missing,"policy_ref":"POL-RCT-02" if missing else ("POL-RCT-01" if required else None)}

def limit_checker(item):
    c=item["category"]; amount=float(item["amount"])
    if c=="meals": limit=75*int(item.get("days",1)); ref="POL-PD-01"
    elif c=="lodging": limit=200*int(item.get("nights",1)); ref="POL-PD-02"
    elif c=="ground_transport": limit=50*int(item.get("days",1)); ref="POL-PD-03"
    else: return {"tool":"limit_checker","approved":amount,"deducted":0,"policy_ref":None}
    return {"tool":"limit_checker","approved":min(amount,limit),"deducted":max(0,amount-limit),"limit":limit,"policy_ref":ref}

def approval_checker(amount):
    if amount<=500: return {"tool":"approval_threshold_checker","tier":"AUTO_APPROVE","manual":False,"policy_ref":"POL-APR-01"}
    if amount<=2000: return {"tool":"approval_threshold_checker","tier":"MANAGER_APPROVAL","manual":False,"policy_ref":"POL-APR-02"}
    return {"tool":"approval_threshold_checker","tier":"MANUAL_REVIEW","manual":True,"policy_ref":"POL-APR-03"}

## 4. Optional LLM Planner

When `OPENAI_API_KEY` is available, the planner can ask an OpenAI-compatible model which tools are needed. Otherwise it uses a deterministic fallback. The final decision is always protected by deterministic policy guardrails.

In [ ]:
LLM_ENABLED=bool(os.getenv("OPENAI_API_KEY"))

def agent_plan(claim):
    tools=["policy_lookup","receipt_checker","limit_checker","approval_threshold_checker"]
    if not LLM_ENABLED: return {"mode":"deterministic_fallback","tools":tools}
    try:
        from openai import OpenAI
        client=OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        prompt=("You are an agent planner for a travel reimbursement system. "
                "Return JSON only with a tools list chosen from policy_lookup, receipt_checker, "
                "limit_checker, approval_threshold_checker. Claim="+json.dumps(claim))
        r=client.chat.completions.create(model=os.getenv("OPENAI_MODEL","gpt-4o-mini"),temperature=0,
            messages=[{"role":"system","content":"Return valid JSON only."},{"role":"user","content":prompt}])
        return {"mode":"llm","tools":json.loads(r.choices[0].message.content)["tools"]}
    except Exception as e:
        return {"mode":"fallback_after_llm_error","tools":tools,"error":str(e)}

print("LLM planner enabled:",LLM_ENABLED)

In [ ]:
Decision=Literal["APPROVE","PARTIAL_APPROVE","REJECT","MANUAL_REVIEW"]
class ClaimDecision(BaseModel):
    claim_id:str
    decision:Decision
    approved_amount:float
    deducted_amount:float
    missing_docs:List[str]
    policy_refs:List[str]
    confidence:float=Field(ge=0,le=1)
    explanation:str
    tools_used:List[str]

## 5. Agentic Orchestrator + Guardrails

The planner selects tools, tools return evidence, and the guardrail layer combines evidence into the final decision. This keeps LLM behavior explainable and prevents the model from overriding policy arithmetic.

In [ ]:
INELIGIBLE={"spa","minibar","alcohol","gym","personal_entertainment","personal_shopping","gifts","traffic_fine","penalty","late_fee"}

def evaluate_claim(claim):
    plan=agent_plan(claim); tools=list(dict.fromkeys(plan["tools"]))
    approved=deducted=0.0; missing=[]; refs=set(); reasons=[]; manual=False; ineligible=False; reimbursable=False; partial=False
    late=(datetime.fromisoformat(claim["submitted"])-datetime.fromisoformat(claim["trip_end"])).days>30
    if late: manual=True; refs.add("POL-TIME-01"); reasons.append("Claim is outside the 30-day submission window.")
    for item in claim["items"]:
        p=policy_lookup(item); refs.update(x["id"] for x in p["policies"])
        c=item["category"]; amt=float(item["amount"])
        if c in INELIGIBLE:
            ineligible=True; refs.add("POL-CAT-02"); deducted+=amt; reasons.append(f"{c} is ineligible under POL-CAT-02."); continue
        reimbursable=True
        rc=receipt_checker(item)
        if rc["policy_ref"]: refs.add(rc["policy_ref"])
        if rc["missing"]:
            manual=True; missing.append(f"{c}: required receipt"); reasons.append(f"Required receipt is missing for {c}.")
        if c=="airfare" and item.get("class","economy").lower() in {"business","first"}:
            manual=True; refs.add("POL-AIR-01"); reasons.append("Business/first-class airfare is a policy exception.")
        lim=limit_checker(item); approved+=lim["approved"]; deducted+=lim["deducted"]
        if lim["deducted"]>0:
            partial=True; refs.add(lim["policy_ref"]); reasons.append(f"{c} exceeds its policy cap by ${lim['deducted']:.2f}.")
    if ineligible and not reimbursable:
        decision="REJECT"; conf=.99; explanation="All claimed items are ineligible. "+" ".join(reasons)
    else:
        ap=approval_checker(approved); refs.add(ap["policy_ref"]); manual=manual or ap["manual"]
        if ap["manual"]: reasons.append(f"Reimbursable amount ${approved:.2f} exceeds agent approval authority.")
        if manual: decision="MANUAL_REVIEW"; conf=.98; explanation="Manual Review required. "+" ".join(reasons)
        elif partial: decision="PARTIAL_APPROVE"; conf=.99; explanation=f"Partially approve ${approved:.2f}; deduct ${deducted:.2f}. "+" ".join(reasons)
        else: decision="APPROVE"; conf=.99; explanation=f"Claim is compliant and approvable for ${approved:.2f}."
    final_approved = 0.0 if decision == "MANUAL_REVIEW" else round(approved, 2)
    final_deducted = 0.0 if decision == "MANUAL_REVIEW" else round(deducted, 2)
    return ClaimDecision(claim_id=claim["claim_id"],decision=decision,approved_amount=final_approved,deducted_amount=final_deducted,missing_docs=missing,policy_refs=sorted(refs),confidence=conf,explanation=explanation,tools_used=tools)

results=[evaluate_claim(c) for c in CLAIMS]

## 6. Expected Outcomes

The five policy-driven outcomes are: **CLM-001 APPROVE, CLM-002 REJECT, CLM-003 PARTIAL_APPROVE, CLM-004 MANUAL_REVIEW, CLM-005 MANUAL_REVIEW.**

In [ ]:
expected={"CLM-001":"APPROVE","CLM-002":"REJECT","CLM-003":"PARTIAL_APPROVE","CLM-004":"MANUAL_REVIEW","CLM-005":"MANUAL_REVIEW"}
validation=pd.DataFrame([{"claim_id":r.claim_id,"actual":r.decision,"expected":expected[r.claim_id],"pass":r.decision==expected[r.claim_id]} for r in results])
display(validation)
assert validation["pass"].all()
print("All policy outcome checks passed.")

## 7. Final Structured Output

Exactly the fields requested by the assignment are emitted for every claim.

In [ ]:
final_json=[r.model_dump() for r in results]
print(json.dumps(final_json,indent=2))

## 8. Dashboard

In [ ]:
df=pd.DataFrame(final_json)
claimed=sum(sum(float(i["amount"]) for i in c["items"]) for c in CLAIMS)
print("=== Travel Reimbursement Agent Dashboard ===")
print(f"Claims: {len(df)} | Claimed: ${claimed:,.2f} | Approved: ${df.approved_amount.sum():,.2f} | Deducted: ${df.deducted_amount.sum():,.2f}")
fig,ax=plt.subplots(1,2,figsize=(12,4))
df.decision.value_counts().plot(kind="bar",ax=ax[0],title="Decision Breakdown")
pd.DataFrame({"USD":[df.approved_amount.sum(),df.deducted_amount.sum()]},index=["Approved","Deducted"]).plot(kind="bar",ax=ax[1],title="Approved vs Deducted Amount",legend=False)
plt.tight_layout(); plt.show()

## 9. Audit Trail

The audit view exposes tool usage, policy references, confidence and the explanation supporting each result.

In [ ]:
display(df[["claim_id","decision","approved_amount","deducted_amount","missing_docs","policy_refs","confidence","tools_used","explanation"]])

## 10. Design Notes & Reasoning

- **Tool calling:** policy retrieval, receipt checking, limit checking and approval-threshold checking are separate tools.
- **Grounding:** policy retrieval supplies stable `POL-*` references before decisions.
- **Reliability:** arithmetic and policy enforcement are deterministic; the LLM is not trusted with final calculations.
- **Manual Review:** used for missing required receipts, business/first-class exceptions, claims over $2,000, late claims or conflicting information.
- **Why no vector DB:** the supplied policy is very small; keyword retrieval is transparent and sufficient for this time-boxed prototype.
- **Production path:** Azure OpenAI, Azure AI Search/vector DB, OCR/Document Intelligence, tracing/evaluation, RBAC, PII controls, policy versioning, CI/CD and human-in-the-loop approval.

## 11. Assumptions & Limitations

- The five supplied claims are the complete demo dataset.
- Currency is USD.
- Provided nights/days are used for category-cap calculations.
- The supplied policy is authoritative.
- No real employee/company data is used.
- This is a prototype, not a production reimbursement platform.
- Thanks HCL for giving me chance to make this project, looking for more continued work with you.